In [265]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from pprint import pprint

llm = ChatOllama(model="llama3.2:3b")

#embedding_model = OllamaEmbeddings(model="nomic-embed-text")
embedding_model = OllamaEmbeddings(model="mxbai-embed-large")


events_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="events",
    embedding_function=embedding_model
)

sections_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="sections",
    embedding_function=embedding_model
)

In [269]:
#query = "What are the technical specs of the Nintendo Switch 2 game console?"
#query = "Tell me the news of Nintendo Switch 2"
query = "What was the market reception of the Nintendo Switch 2 game console?"
#query = "Tell me about Nintendo gaming consoles"
#query = "Compare the features of Nintendo Switch 2 to the original Nintendo Switch"
#query = "What are achievements of Prema Racing?"

query_vector = embedding_model.embed_query(query)

top_events = events_vector_store.similarity_search_by_vector(
    query_vector, 
    k=5
)

event_index = {}
section_index = {}

#pprint(top_events)

formatted_events = ""

for event in top_events:
    #print(event)
    event_index[event.id] = event
    # don't use the question and reinsert the date
    event.page_content = event.page_content.split('?\n\n', 1)[1]  
    event.page_content = f"{event.metadata['day']} {event.metadata['month']}, {event.metadata['year']}: {event.page_content}"
    formatted_events += f"\n\n\nEvent {event.id}: {event.page_content}"
    top_sections = sections_vector_store.similarity_search_by_vector(
        query_vector,
        k=5,
        ids=event.metadata["sections"]
    )
    for section in top_sections:
        formatted_events += f"\n\t{event.id}.{section.id}: {section.metadata["title"]}"
        section_index[section.id]=section
    

select_events_prompt = ChatPromptTemplate.from_template( 
    """
    I was this query by the user: {query}

    These events and their sections could be relevant but some of them are not relevant:
        {formatted_events}

    Select only the events relevant to the user's query using a maximum of 5 events and 30 sections. Skip events not directly relevant to the user's query.
    
    Give the output in a JSON format strictly like this example: {{"e115": ["s899", "s655"], "e782": ["s311"]}}

    Only use event IDs in keys and nothing else.
    Only use section IDs for section elements and nothing else.
    
    Just output one object in the JSON and nothing else! 
    """
)

filled_select_events_prompt = select_events_prompt.format(
    query = query,
    formatted_events = formatted_events
)

# print(filled_select_events_prompt)

print(f"\nSelect events prompt length: {len(filled_select_events_prompt)}\n\n")


chain = select_events_prompt | llm | StrOutputParser()
json_str = chain.invoke({
    "query": query,
    "formatted_events": formatted_events
})

clean_json = json_str[json_str.find("{"):json_str.rfind("}") + 1]

print(f"JSON: {clean_json}")

import json

retrieved_content = ""

try:
    data = json.loads(clean_json)
    for ei, (event_id, sections) in enumerate(data.items(), 1):
        event = event_index.get(event_id)
        if event is None: 
            continue
            
        retrieved_content += f"{event.page_content} "
        for ej, section_id in enumerate(sections, 1):
            section_id = section_id.split(".", 1)[-1] # sometimes the format returned is e12313.s2343
            section = section_index.get(section_id)
            if section is None:
                continue
            retrieved_content += f"{section.metadata['summary']} "
        retrieved_content += "\n\n"
    
    print("\nRetrieved: \n")
    print(retrieved_content)

except Exception:
    data = None
    retrieved_content = "No articles found"


final_prompt = ChatPromptTemplate.from_template( 
    """
    I was asked this query by the user: {query}

    Answer the user's query using only relevant parts of the below events that may not be connected to each other:
    {retrieved_content}

    Add factual information from your own knowledge-base to enhance the provided events but do not speculate.
    If no relevant events are listed, use your own knowledge-base to answer. Ignore events that are not connected.
    Just answer without mentioning news was provided to you but use the news.
    """
)

filled_final_prompt = final_prompt.format(
    query =  query,
    retrieved_content = retrieved_content
)

print(f"\nFinal prompt length: {len(filled_final_prompt)}\n\n")

chain = final_prompt | llm | StrOutputParser()

output = chain.invoke({
    "query": query,
    "retrieved_content": retrieved_content
})

print("\n\nFinal output:\n")
print(output)


Select events prompt length: 3012


JSON: {"e9": ["s88", "s90"], "e20": ["s176", "s177"], "e74": ["s550", "s552"]}

Retrieved: 

1 January, 2024: Egypt, Ethiopia, Iran, Saudi Arabia, and the United Arab Emirates formally join the BRICS group as new members. (Tehran Times) The formal addition of five new countries to the BRICS group on January 1, 2024, significantly expanded the group's geopolitical weight, economic structure, and influence. The BRICS-10 bloc now dominates the global economy, accounting for nearly half of the world's population and over 35% of its total GDP, with emerging infrastructure financing institutions like the New Development Bank poised to challenge traditional Western-dominated multilateral financial institutions. 

2 January, 2024: 2024 Haneda Airport runway collision
A Japan Airlines Airbus A350-900 aircraft collides with a Japan Coast Guard Dash 8 aircraft and bursts into flames at Haneda Airport in Tokyo, Japan. All 379 occupants aboard the Japan Airlines